# Practical 3: Design Toy Application Using Blockchain
## Student Certificate Verification System

**Concepts demonstrated:** Blocks, SHA-256 hashing, previous hash, nonce, simple Proof-of-Work mining, blockchain construction, validation, and tampering detection.

In [5]:
import hashlib #built-in python module which provides functions for creating hash.
from datetime import datetime # datetime needed for timestamp

## Part 1: Block Class

A block stores the certificate information and the information required to connect it with the blockchain.

Each block contains:
- Certificate data
- Timestamp
- Previous hash
- Nonce
- Current hash

In [13]:
class Block:

    # Constructor: runs automatically when a new block is created
    def __init__(self, data, previous_hash):
      
        self.data = data # Store student/certificate information     
        self.timestamp = datetime.now() # Store the time when the block is created       
        self.previous_hash = previous_hash # Store the hash of the previous block      
        self.nonce = 0 # Nonce is changed during mining  
        self.hash = self.calculate_hash() # Calculate the initial hash of this block

    # Calculate the SHA-256 hash of the block
    def calculate_hash(self):
        # The hash depends on the block's data, timestamp,
        # previous hash, and nonce.
        block_data = (
            str(self.data)
            + str(self.timestamp)
            + self.previous_hash
            + str(self.nonce)
        )
        
        # Convert text to bytes, calculate SHA-256,
        # and return the readable hexadecimal hash.
        return hashlib.sha256(block_data.encode()).hexdigest()

    # Simple Proof-of-Work mining
    def mine_block(self, difficulty):       
       
        target = "0" * difficulty # difficulty = 4 means the hash must start with 0000
        # Keep changing the nonce until the required hash is found
        while not self.hash.startswith(target):

            # Try the next nonce
            self.nonce += 1

            # Calculate a new hash
            self.hash = self.calculate_hash()

        print("Block mined successfully!")

## Part 2: Blockchain Class

The `Blockchain` class manages the complete chain. It creates the Genesis Block, adds certificate blocks, mines them, and checks whether the chain has been modified.

In [7]:
class Blockchain:

    def __init__(self):
        
        self.chain = [] # List that stores all blocks
        self.difficulty = 4 # Four leading zeros are required for mining      
        self.create_genesis_block() # Create the first block

    # Create the first block of the blockchain
    def create_genesis_block(self):

        # Genesis Block has no real previous block
        genesis_block = Block("Genesis Block","0000000000000000")

        genesis_block.mine_block(self.difficulty) # Mine the Genesis Block
        self.chain.append(genesis_block) # Add it to the chain

    # Return the latest block
    def get_latest_block(self):
        return self.chain[-1]

    # Add a new certificate block
    def add_block(self, data):
        
        latest_block = self.get_latest_block() # Get the latest block
        new_block = Block(data,latest_block.hash) # The new block stores the previous block's hash
        new_block.mine_block(self.difficulty) # Mine the new block       
        self.chain.append(new_block) # Add it to the blockchain

    # Check whether the complete chain is valid
    def is_chain_valid(self):

        # Start from Block 1 because Block 0 is the Genesis Block
        for i in range(1, len(self.chain)):

            current_block = self.chain[i]
            previous_block = self.chain[i - 1]

            # Check whether the current block's data was changed
            if current_block.hash != current_block.calculate_hash():
                return False

            # Check whether the blocks are still correctly linked
            if current_block.previous_hash != previous_block.hash:
                return False

        return True

## Part 3: Create the Blockchain

Creating the `Blockchain` object automatically creates and mines the Genesis Block.

In [8]:
# Create a new blockchain
certificate_blockchain = Blockchain()

Block mined successfully!


## Part 4: Add Student Certificates

Each certificate is stored as the data of a blockchain block.

In [9]:
# Certificate of Student 1
certificate1 = {
    "student": "Rahul Sharma",
    "roll_no": 101,
    "course": "BTech",
    "certificate": "Python Workshop",
    "year": 2026
}

certificate_blockchain.add_block(certificate1)


# Certificate of Student 2
certificate2 = {
    "student": "Priya Singh",
    "roll_no": 102,
    "course": "BTech",
    "certificate": "Machine Learning Workshop",
    "year": 2026
}

certificate_blockchain.add_block(certificate2)


# Certificate of Student 3
certificate3 = {
    "student": "Aman Kumar",
    "roll_no": 103,
    "course": "BTech",
    "certificate": "Blockchain Workshop",
    "year": 2026
}

certificate_blockchain.add_block(certificate3)

Block mined successfully!
Block mined successfully!
Block mined successfully!


## Part 5: Display the Blockchain

The following code displays the important information stored in every block.

In [10]:
print("\n========== BLOCKCHAIN ==========")

for i, block in enumerate(certificate_blockchain.chain):

    print("\nBlock Number:", i)
    print("Data:", block.data)
    print("Timestamp:", block.timestamp)
    print("Previous Hash:", block.previous_hash)
    print("Nonce:", block.nonce)
    print("Hash:", block.hash)
    print("--------------------------------")


========== BLOCKCHAIN ==========

Block Number: 0
Data: Genesis Block
Timestamp: 2026-08-17 21:23:17.886955
Previous Hash: 0000000000000000
Nonce: 210261
Hash: 0000ad28bdc498d31adb746cfa78abf6e39788c642d209eebb4b5d0ce19d1510
--------------------------------

Block Number: 1
Data: {'student': 'Rahul Sharma', 'roll_no': 101, 'course': 'BCA', 'certificate': 'Python Workshop', 'year': 2026}
Timestamp: 2026-08-17 21:23:19.195356
Previous Hash: 0000ad28bdc498d31adb746cfa78abf6e39788c642d209eebb4b5d0ce19d1510
Nonce: 7650
Hash: 000002bb2b9f6c9d6945d5759f22db37a0cc5b15c7d869dfc7e95fdbab833502
--------------------------------

Block Number: 2
Data: {'student': 'Priya Singh', 'roll_no': 102, 'course': 'BCA', 'certificate': 'Machine Learning Workshop', 'year': 2026}
Timestamp: 2026-08-17 21:23:19.264034
Previous Hash: 000002bb2b9f6c9d6945d5759f22db37a0cc5b15c7d869dfc7e95fdbab833502
Nonce: 113166
Hash: 000057f66c099fab0680365598b07f696071ee939341140533167c78d8c8e1f9
-------------------------------

## Part 6: Verify the Blockchain

The chain is valid when:
1. The stored hash of every block is still correct.
2. Each block's `previous_hash` matches the actual hash of the previous block.

In [11]:
print("\nChecking blockchain validity...")

if certificate_blockchain.is_chain_valid():
    print("Blockchain is VALID.")
else:
    print("Blockchain is INVALID.")


Checking blockchain validity...
Blockchain is VALID.


## Part 7: Demonstrate Tampering

We now intentionally change Rahul's certificate from `BCA` to `MCA`.

This simulates someone trying to modify a certificate after it has been stored in the blockchain.

In [12]:
print("\n========== TAMPERING TEST ==========")

# Intentionally change certificate data in Block 1
certificate_blockchain.chain[1].data["course"] = "MCA"

print("Blockchain validity after changing certificate data:")

if certificate_blockchain.is_chain_valid():
    print("Blockchain is VALID.")
else:
    print("Blockchain is INVALID.")
    print("Tampering has been detected!")


========== TAMPERING TEST ==========
Blockchain validity after changing certificate data:
Blockchain is INVALID.
Tampering has been detected!


## Expected Result

Before tampering:

```text
Blockchain is VALID.
```

After changing the certificate:

```text
Blockchain is INVALID.
Tampering has been detected!
```

### Complete Flow

```text
Student Certificate
        ↓
      Block
        ↓
    SHA-256 Hash
        ↓
     Mining
        ↓
  Add to Blockchain
        ↓
 Verify Blockchain
        ↓
 Detect Tampering
```